# Lab L2.3 · Income estimation — gradient descent, hyperparameter search and honest model selection
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 2 · ⏱ 60 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Estimate monthly income for customers whose salary is not paid into HGB: compare grid, random and successive-halving search, measure tuning optimism with nested cross-validation, and hand Credit a conservative income figure.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

UAE affordability rules cap a borrower's **debt-burden ratio** at 50% of income. For customers whose salary is transferred to HGB the income is verified; for the other ~45% Credit relies on self-declared figures and paper certificates. Credit Risk asks MIZAN for an **income estimation model** trained on verified customers and applied to the rest — with an honest error estimate and a **conservative** figure that can be used in affordability checks.

**Progressive architecture:** uses L1.1's clean master and L1.3's transaction features; income estimates can enrich credit decisions (Day 3) and the capstone (Day 5).

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Assemble the data — and notice who is missing

**Why this matters.** The model learns from customers with **verified** salaries and is applied to customers *without* them. If those two groups differ (they do: self-employed people rarely transfer salary), the model is being asked to **extrapolate**. Spotting this **selection bias** before modelling is part of the job.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — side-by-side shares expose the selection bias

In [ ]:
import json
cust = pd.read_parquet(CKPT / "customers_clean.parquet")                    # L1.1
txf = pd.read_parquet(CKPT / "customer_txn_features.parquet")              # L1.3
spec = json.loads((CKPT / "txn_feature_spec.json").read_text())
TXN = spec["selected"] + ["has_card_activity"]
NUM = ["age", "tenure_months", "num_products", "has_credit_card", "has_personal_loan", "has_auto_loan", "has_mortgage",
       "avg_balance_aed", "card_spend_monthly_aed", "digital_logins_30d", "branch_visits_90d", "aecb_score"] + TXN
CAT = ["employment_type", "employer_category", "emirate"]
df = cust.merge(txf[["customer_id"] + TXN], on="customer_id", how="left", validate="one_to_one")
df[["avg_balance_aed", "card_spend_monthly_aed"]] = np.log1p(df[["avg_balance_aed", "card_spend_monthly_aed"]])
verified = df[(df.salary_transfer == 1) & (df.salary_missing == 0)].copy()
unverified = df[df.salary_transfer == 0].copy()
verified["y"] = np.log(verified.monthly_salary_aed)
print(f"verified (train/test): {len(verified):,} | unverified (to score): {len(unverified):,} | features: {len(NUM)} numeric + {len(CAT)} categorical")
mix = pd.concat({"verified": verified.employment_type.value_counts(normalize=True),
                 "unverified": unverified.employment_type.value_counts(normalize=True)}, axis=1)
____  # ✏️ B1

**✔ Checkpoint.** ≈ 147k verified vs ≈ 97k unverified customers. Self-employed are ≈ 3% of verified but ≈ 27% of unverified — a strong selection bias.

## Step 2 · Gradient descent by hand: learning rate, batch size, epochs

**Why this matters.** Every neural network and `SGDRegressor` learns by **iterative optimisation**: take a small batch, compute the gradient of the loss, step downhill by *learning rate × gradient*, repeat for several **epochs**. Watching the loss curve tells you whether the learning rate is too small (slow), right, or too large (divergence).

#### ✏️ What to write in the next cell
- **Blank 2** *(inline)* — gradient of the mean squared error w.r.t. the weights
- **Blank 3** *(inline)* — one step downhill

In [ ]:
from sklearn.linear_model import LinearRegression
cols = ["age", "tenure_months", "avg_balance_aed", "card_spend_monthly_aed", "aecb_score"]
Xg = verified[cols].fillna(verified[cols].median()).to_numpy(); Xg = (Xg - Xg.mean(0)) / Xg.std(0)
yg = (verified.y - verified.y.mean()).to_numpy()

def minibatch_gd(X, y, lr, batch=256, epochs=10, seed=RNG):
    rng, w, hist = np.random.default_rng(seed), np.zeros(X.shape[1]), []
    for _ in range(epochs):
        order = rng.permutation(len(y))
        for i in range(0, len(y), batch):
            b = order[i:i + batch]
            grad = ____  # ✏️ B2
            w = ____  # ✏️ B3
        hist.append(np.mean((X @ w - y) ** 2))
    return w, hist

for lr in [0.001, 0.01, 0.1, 1.5]:
    w, hist = minibatch_gd(Xg, yg, lr)
    plt.plot(range(1, 11), np.clip(hist, 0, 1), marker="o", label=f"lr={lr}")
plt.xlabel("epoch"); plt.ylabel("training MSE (clipped at 1)"); plt.legend(); plt.title("Mini-batch gradient descent"); plt.show()
w_gd, _ = minibatch_gd(Xg, yg, 0.01)
print("GD weights     :", w_gd.round(3)); print("closed form OLS:", LinearRegression().fit(Xg, yg).coef_.round(3))

**✔ Checkpoint.** lr 0.001 converges slowly, 0.01–0.1 converge within 1–2 epochs, 1.5 diverges (loss shoots off the chart). GD weights match the closed-form solution to ≈ 2–3 decimals.

## Step 3 · Baseline, then a regularised pipeline

**Why this matters.** A model is only worth deploying if it beats what Credit does today: **median salary for the applicant's employment type and employer category**. We hold out a test set first, then compare that rule with a Ridge pipeline on all features. Errors are reported in **percent** (MAPE on the original scale) because a AED 2,000 miss matters more at AED 8k than at AED 80k.

#### ✏️ What to write in the next cell
- **Blank 4** *(inline)* — look up each test customer's segment median

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_percentage_error as mape, r2_score
tr, te = train_test_split(verified, test_size=0.2, random_state=RNG)
rule = tr.groupby(["employment_type", "employer_category"], observed=True).monthly_salary_aed.median()
base_pred = ____  # ✏️ B4
prep = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
ridge = make_pipeline(prep, RidgeCV(alphas=np.logspace(-2, 3, 12))).fit(tr[NUM + CAT], tr.y)
p_ridge = np.exp(ridge.predict(te[NUM + CAT]))
board = pd.DataFrame({"MAPE": [mape(te.monthly_salary_aed, base_pred), mape(te.monthly_salary_aed, p_ridge)],
                      "R2_log": [r2_score(te.y, np.log(np.asarray(base_pred, float))), r2_score(te.y, np.log(p_ridge))]},
                     index=["segment-median rule", "ridge pipeline"])
board.round(3)

**✔ Checkpoint.** Segment-median rule: MAPE ≈ 51%, R² ≈ 0.12. Ridge pipeline: clearly better — MAPE ≈ 34%, R² ≈ 0.59 on the log scale.

## Step 4 · Search shoot-out: grid vs random vs successive halving

**Why this matters.** A gradient-boosting regressor has many **hyperparameters** (learning rate, depth, leaf size, L2). **Grid search** tries every combination — exhaustive but expensive. **Random search** samples combinations and usually finds an equally good one far faster. **Successive halving** gives every candidate a small budget and only promotes the best to more data. We compare **best CV score and wall-clock time** on the same 40k-row training sample.

#### ✏️ What to write in the next cell
- **Blank 5** *(inline)* — every combination × 3 folds
- **Blank 6** *(inline)* — 12 random draws

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, HalvingRandomSearchCV, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from scipy.stats import loguniform, randint
prep_t = ColumnTransformer([("num", "passthrough", NUM),
                            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT)])
hgb = make_pipeline(prep_t, HistGradientBoostingRegressor(max_iter=200, early_stopping=False, random_state=RNG))
S = tr.sample(40_000, random_state=RNG); XS, yS = S[NUM + CAT], S.y
cv = KFold(3, shuffle=True, random_state=RNG)
grid = {"histgradientboostingregressor__learning_rate": [0.05, 0.1, 0.2], "histgradientboostingregressor__max_leaf_nodes": [15, 31, 63],
        "histgradientboostingregressor__min_samples_leaf": [20, 100], "histgradientboostingregressor__l2_regularization": [0, 1]}
dist = {"histgradientboostingregressor__learning_rate": loguniform(0.03, 0.3), "histgradientboostingregressor__max_leaf_nodes": randint(8, 80),
        "histgradientboostingregressor__min_samples_leaf": randint(10, 200), "histgradientboostingregressor__l2_regularization": loguniform(1e-3, 10)}
searches = {"grid (36)": ____,  # ✏️ B5
            "random (12)": ____,  # ✏️ B6
            "halving (36→12→4)": HalvingRandomSearchCV(hgb, dist, n_candidates=36, factor=3, resource="histgradientboostingregressor__max_iter",
                                                        min_resources=22, max_resources=200, cv=cv, scoring="r2", n_jobs=-1, random_state=RNG)}
rows = {}
for name, s in searches.items():
    t0 = time.time(); s.fit(XS, yS)
    fits = len(s.cv_results_["params"]) * 3
    rows[name] = {"best_cv_R2": s.best_score_, "fits": fits, "seconds": time.time() - t0}
shoot = pd.DataFrame(rows).T; display(shoot.round(3))
best = searches["random (12)"].best_params_
{k.split("__")[1]: round(float(v), 3) for k, v in best.items()}

**✔ Checkpoint.** Grid and random search tie at CV R² ≈ 0.650 — random needs a third of the fits and ≈ ¼ of the time (≈ 5 s vs ≈ 19 s). Halving screens all 36 candidates in under half the grid's time and lands within 0.005. Boosting (≈ 0.65) clearly beats Ridge (≈ 0.59).

## Step 5 · Nested cross-validation — how optimistic was that score?

**Why this matters.** `best_score_` is the **best of many tries**, so it is optimistically biased — like picking the luckiest of 36 coin-tossers. **Nested CV** wraps the whole search inside an outer CV: each outer fold tunes on its own inner folds and is scored on data the search never saw. The outer mean is an honest estimate of *the whole procedure*.

#### ✏️ What to write in the next cell
- **Blank 7** *(inline)* — the search is re-run inside every outer fold

In [ ]:
from sklearn.model_selection import cross_val_score
S2 = tr.sample(15_000, random_state=RNG + 1)
inner = RandomizedSearchCV(hgb, dist, n_iter=8, cv=KFold(3, shuffle=True, random_state=RNG), scoring="r2", n_jobs=-1, random_state=RNG)
outer = ____  # ✏️ B7
inner.fit(S2[NUM + CAT], S2.y)
print(f"inner best_score_ (optimistic): {inner.best_score_:.4f} | nested outer mean: {outer.mean():.4f} ± {outer.std():.4f}")

**✔ Checkpoint.** Both ≈ 0.63–0.64 and within one standard deviation of each other: with plenty of data and similar candidates the optimism is negligible. That is a *finding*, not a given — on small data with large grids the inner score can overstate performance badly.

## Step 6 · Final test, error analysis and selection bias

**Why this matters.** Refit the chosen configuration on the full training set and touch the **test set once**. Then break the error down by segment: a single MAPE hides that some groups (self-employed, the very rich) are much harder.

#### ✏️ What to write in the next cell
- **Blank 8** *(inline)* — salary quintiles

In [ ]:
final = hgb.set_params(**best).fit(tr[NUM + CAT], tr.y)
pred = np.exp(final.predict(te[NUM + CAT]))
err = np.abs(pred / te.monthly_salary_aed - 1)
print(f"TEST  MAPE {err.mean():.1%} | R² (log) {r2_score(te.y, np.log(pred)):.3f} | within ±20%: {(err <= 0.2).mean():.1%}")
seg = te.assign(ape=err, band=____)  # ✏️ B8
display(seg.groupby("employment_type", observed=True).ape.agg(["mean", "size"]).round(3))
seg.groupby("band", observed=True).ape.mean().round(3)

**✔ Checkpoint.** Test MAPE ≈ 31% (rule 51%), R² ≈ 0.65, ≈ 42% of customers within ±20%. Errors are similar across employment types *inside* the verified group (only ≈ 900 self-employed to learn from), but much higher in the bottom quintile (≈ 52%) where the model over-estimates — regression to the mean.

## Step 7 · Conservative income for unverified customers

**Why this matters.** Credit needs a figure they can defend. Instead of the point estimate, use a **conservative** one: shift each prediction down by the 20th percentile of the test residuals (log scale), so ≈ 80% of verified customers earn at least that much. Save the model with its residual quantile so the rule is reproducible.

#### ✏️ What to write in the next cell
- **Blank 9** *(inline)* — log-residual below which 20% of verified customers fall

In [ ]:
import joblib
q20 = ____  # ✏️ B9
print(f"coverage check on test: {(te.monthly_salary_aed >= pred * np.exp(q20)).mean():.1%} earn ≥ the conservative figure")
est = np.exp(final.predict(unverified[NUM + CAT]))
out = pd.DataFrame({"customer_id": unverified.customer_id, "income_estimate_aed": est.round(-1),
                    "income_conservative_aed": (est * np.exp(q20)).round(-1), "declared_salary_aed": unverified.monthly_salary_aed})
out.to_parquet(CKPT / "income_estimates.parquet", index=False)
joblib.dump({"model": final, "features": NUM + CAT, "log_q20": float(q20), "test_mape": float(err.mean())}, CKPT / "income_model.joblib")
print(f"q20 = {q20:.3f} (× {np.exp(q20):.2f}) | unverified: median declared {out.declared_salary_aed.median():,.0f} vs "
      f"estimate {out.income_estimate_aed.median():,.0f} vs conservative {out.income_conservative_aed.median():,.0f} AED")

**✔ Checkpoint.** Coverage 80% by construction; the conservative figure is ≈ 0.74× the estimate. For unverified customers the median declared salary (≈ 12,800) is close to the median estimate (≈ 12,400); the conservative median is ≈ 9,100 AED. Files saved.

## ✅ Lab complete
Credit has an income estimate with an honest error and a conservative figure. **Tomorrow:** classification — default, segments and fraud.